# Unlearning Experiments on HR, PACS, and BloodMNIST

This notebook contains only the unlearning part of the project. It trains a `FedAvg` base model for each dataset and then applies some unlearning methods in federated settings to compare forgetting, retention, and OOD utility.

## What is included

- dataset setup for `HR`, `PACS`, and `BloodMNIST`
- `FedAvg` base training with saved round history
- unlearning methods: `Split`, `NPO`, `GA`, `Amnesiac`, and `Zero-shot`
- export of all unlearning results to Excel

In [1]:
import copy
import gc
import random
import time
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from torch.utils.data import ConcatDataset, DataLoader, TensorDataset, Subset
from torchvision import datasets as tv_datasets, models as tv_models, transforms
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

DEVICE

device(type='mps')

In [2]:
HR_DATA_PATH = Path("data/Hr/stackoverflow_full.csv")
PACS_ROOT = Path("data/pacs")
PACS_DATASET = "pacs"
MEDMNIST_DATASET = "bloodmnist"
PACS_DOMAINS = ["art_painting", "cartoon", "photo", "sketch"]
PACS_OOD_DOMAIN = "sketch"

CONFIG = {
    "batch_size": 64,
    "image_batch_size": 48,
    "hidden_dim": 128,
    "lr": 1e-3,
    "image_lr": 3e-4,
    "image_lr_candidates": [3e-4, 1e-3],
    "image_lr_search_epochs": 2,
    "weight_decay": 1e-4,
    "local_epochs": 3,
    "image_local_epochs": 5,
    "rounds": 10,
    "image_centralized_epochs": 40,
    "num_clients": 3,
    "fedprox_mu": 1e-2,
    "fedproto_lambda": 0.5,
    "dp_clip": 1.0,
    "dp_noise_multiplier": 0.08,
    "mixup_alpha": 0.2,
    "label_smoothing": 0.1,
    "image_dropout": 0.3,
    "centralized_patience": 8,
    "dirichlet_alpha": 0.6,
    "dirichlet_min_size": 32,
}

print("Device:", DEVICE)
print("HR path exists:", HR_DATA_PATH.exists() or (Path("core") / HR_DATA_PATH).exists())
print("PACS root:", PACS_ROOT)
print("BloodMNIST subset:", MEDMNIST_DATASET)

Device: mps
HR path exists: True
PACS root: data/pacs
BloodMNIST subset: bloodmnist


## Dataset setup

- `HR`: country-shift split for IND and OOD evaluation
- `PACS`: training on `art_painting`, `cartoon`, and `photo`, with `sketch` used for OOD testing
- `BloodMNIST`: original test split for IND and a rotated plus brightness-shifted version for OOD

In [3]:
def release_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if hasattr(torch, "mps") and hasattr(torch.mps, "empty_cache"):
        torch.mps.empty_cache()


def load_hr_dataset(csv_path: Path):
    if not csv_path.exists():
        alt_path = Path("core") / csv_path
        csv_path = alt_path if alt_path.exists() else csv_path
    if not csv_path.exists():
        raise FileNotFoundError(f"HR dataset not found at {csv_path}")

    df = pd.read_csv(csv_path)
    df.columns = [str(c).strip() for c in df.columns]
    if "Unnamed: 0" in df.columns:
        df = df.drop(columns=["Unnamed: 0"])

    target_col = "Employed"
    df = df.dropna(axis=0, how="all").dropna(axis=1, how="all")
    X = df.drop(columns=[target_col]).copy()
    y = df[target_col].copy()
    return X, y


def build_preprocessor(X_train: pd.DataFrame):
    numeric_cols = X_train.select_dtypes(include=[np.number, "bool"]).columns.tolist()
    categorical_cols = [c for c in X_train.columns if c not in numeric_cols]

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_pipe, numeric_cols),
        ("cat", categorical_pipe, categorical_cols),
    ])
    return preprocessor, numeric_cols, categorical_cols


def prepare_hr_dataset_country_shift(X: pd.DataFrame, y, domain_col="Country", test_size=0.2, num_ood_countries=3, min_country_samples=1500):
    df = X.copy()
    y_series = pd.Series(y).copy()
    label_encoder = LabelEncoder()
    df["_target"] = label_encoder.fit_transform(y_series.astype(str))

    counts = df[domain_col].value_counts()
    eligible = counts[counts >= min_country_samples]
    preferred_drop = "United States of America"
    if preferred_drop in eligible.index and len(eligible) > num_ood_countries:
        eligible = eligible.drop(preferred_drop)
    if len(eligible) < num_ood_countries:
        eligible = counts.drop(preferred_drop, errors="ignore")
    ood_countries = eligible.index[:num_ood_countries].tolist()

    seen_df = df[~df[domain_col].isin(ood_countries)].copy()
    ood_df = df[df[domain_col].isin(ood_countries)].copy()
    stratify = seen_df["_target"] if len(np.unique(seen_df["_target"])) > 1 else None
    train_df, ind_df = train_test_split(seen_df, test_size=test_size, random_state=SEED, stratify=stratify)

    X_train_raw = train_df.drop(columns=["_target"])
    X_ind_raw = ind_df.drop(columns=["_target"])
    X_ood_raw = ood_df.drop(columns=["_target"])
    y_train = train_df["_target"].to_numpy(dtype=np.int64)
    y_ind = ind_df["_target"].to_numpy(dtype=np.int64)
    y_ood = ood_df["_target"].to_numpy(dtype=np.int64)

    preprocessor, numeric_cols, categorical_cols = build_preprocessor(X_train_raw)
    X_train_proc = preprocessor.fit_transform(X_train_raw)
    X_ind_proc = preprocessor.transform(X_ind_raw)
    X_ood_proc = preprocessor.transform(X_ood_raw)
    if hasattr(X_train_proc, "toarray"):
        X_train_proc = X_train_proc.toarray()
        X_ind_proc = X_ind_proc.toarray()
        X_ood_proc = X_ood_proc.toarray()

    return {
        "X_train": X_train_proc.astype(np.float32),
        "X_test": X_ind_proc.astype(np.float32),
        "X_ind_test": X_ind_proc.astype(np.float32),
        "X_ood_test": X_ood_proc.astype(np.float32),
        "y_train": y_train,
        "y_test": y_ind,
        "y_ind_test": y_ind,
        "y_ood_test": y_ood,
        "input_dim": X_train_proc.shape[1],
        "model_type": "tabular",
        "num_classes": int(len(np.unique(y_train))),
        "label_encoder": label_encoder,
        "preprocessor": preprocessor,
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols,
        "ood_countries": ood_countries,
    }


def load_medmnist_dataset(dataset_name="bloodmnist"):
    import medmnist
    from medmnist import INFO

    info = INFO[dataset_name]
    data_class = getattr(medmnist, info["python_class"])
    train_ds = data_class(split="train", download=True)
    val_ds = data_class(split="val", download=True)
    test_ds = data_class(split="test", download=True)

    def split_to_numpy(ds):
        images = np.asarray(ds.imgs, dtype=np.float32) / 255.0
        if images.ndim == 3:
            images = images[:, None, :, :]
        elif images.ndim == 4 and images.shape[-1] in (1, 3):
            images = np.transpose(images, (0, 3, 1, 2))
        labels = np.asarray(ds.labels).reshape(-1).astype(np.int64)
        return images.astype(np.float32), labels

    def make_ood_shift(images):
        shifted = np.rot90(images, k=1, axes=(-2, -1)).copy()
        shifted = np.clip(shifted * 0.75 + 0.10, 0.0, 1.0)
        return shifted.astype(np.float32)

    X_train, y_train = split_to_numpy(train_ds)
    X_val, y_val = split_to_numpy(val_ds)
    X_test, y_test = split_to_numpy(test_ds)

    return {
        "X_train": X_train,
        "X_val": X_val,
        "y_val": y_val,
        "X_test": X_test,
        "X_ind_test": X_test,
        "X_ood_test": make_ood_shift(X_test),
        "y_train": y_train,
        "y_test": y_test,
        "y_ind_test": y_test,
        "y_ood_test": y_test.copy(),
        "input_dim": int(np.prod(X_train.shape[1:])),
        "input_channels": int(X_train.shape[1]),
        "image_size": tuple(X_train.shape[2:]),
        "num_classes": int(len(np.unique(y_train))),
        "model_type": "image",
    }


def _resolve_pacs_root(root_dir: Path):
    candidates = [
        root_dir,
        Path.cwd() / root_dir,
        Path.cwd().parent / root_dir,
        Path("core") / root_dir,
        Path("..") / root_dir,
        Path("/Users/shashank.ajmani/Desktop/NTU Assigments/Data Privacy/Learning/DomainFL") / root_dir,
    ]
    for candidate in candidates:
        candidate = Path(candidate)
        if candidate.exists():
            return candidate
    return root_dir


def _stratified_index_split(labels, test_size, seed=SEED):
    idx = np.arange(len(labels))
    train_idx, test_idx = train_test_split(idx, test_size=test_size, random_state=seed, stratify=np.asarray(labels))
    return np.array(sorted(train_idx)), np.array(sorted(test_idx))


def load_pacs_dataset(root_dir: Path, ood_domain="sketch", domains=None):
    root_dir = _resolve_pacs_root(root_dir)
    domains = list(PACS_DOMAINS if domains is None else domains)
    missing = [d for d in domains if not (root_dir / d).exists()]
    if missing:
        raise FileNotFoundError(f"PACS folders not found under {root_dir}. Missing: {missing}")

    seen_domains = [d for d in domains if d != ood_domain]
    weights = tv_models.ResNet18_Weights.DEFAULT
    normalize = transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        normalize,
    ])
    eval_transform = weights.transforms()

    probe_dataset = tv_datasets.ImageFolder(root_dir / seen_domains[0])
    class_names = list(probe_dataset.classes)

    client_subsets = []
    client_domains = []
    train_parts = []
    ind_parts = []
    y_train_all = []

    for domain in seen_domains:
        train_domain_ds = tv_datasets.ImageFolder(root_dir / domain, transform=train_transform)
        eval_domain_ds = tv_datasets.ImageFolder(root_dir / domain, transform=eval_transform)
        train_idx, ind_idx = _stratified_index_split(train_domain_ds.targets, test_size=0.2, seed=SEED)
        train_subset = Subset(train_domain_ds, train_idx.tolist())
        ind_subset = Subset(eval_domain_ds, ind_idx.tolist())
        client_subsets.append(train_subset)
        client_domains.append(domain)
        train_parts.append(train_subset)
        ind_parts.append(ind_subset)
        y_train_all.extend(np.asarray(train_domain_ds.targets)[train_idx].tolist())

    ood_eval_ds = tv_datasets.ImageFolder(root_dir / ood_domain, transform=eval_transform)
    ood_val_idx, ood_test_idx = _stratified_index_split(ood_eval_ds.targets, test_size=0.5, seed=SEED)
    ood_val_subset = Subset(ood_eval_ds, ood_val_idx.tolist())
    ood_test_subset = Subset(ood_eval_ds, ood_test_idx.tolist())

    train_dataset = ConcatDataset(train_parts)
    ind_test_dataset = ConcatDataset(ind_parts)
    sample_x, _ = train_dataset[0][:2]

    return {
        "model_type": "image",
        "num_classes": len(class_names),
        "input_channels": int(sample_x.shape[0]),
        "image_size": tuple(sample_x.shape[1:]),
        "input_dim": int(np.prod(sample_x.shape)),
        "train_dataset": train_dataset,
        "ind_test_dataset": ind_test_dataset,
        "ood_val_dataset": ood_val_subset,
        "ood_test_dataset": ood_test_subset,
        "client_subsets": client_subsets,
        "client_groups": client_domains,
        "y_train": np.asarray(y_train_all, dtype=np.int64),
        "class_names": class_names,
        "ood_domain": ood_domain,
        "seen_domains": seen_domains,
    }

In [4]:
def unpack_batch(batch):
    if isinstance(batch, (list, tuple)) and len(batch) == 3:
        return batch[0], batch[1], batch[2]
    if isinstance(batch, (list, tuple)) and len(batch) == 2:
        return batch[0], batch[1], None
    raise ValueError("Unexpected batch format")


def make_loader(X, y=None, batch_size=256, shuffle=True):
    if y is None:
        ds = X
    else:
        ds = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


def get_batch_size(prepared, cfg):
    if prepared.get("model_type") == "image":
        return cfg.get("image_batch_size", cfg["batch_size"])
    return cfg["batch_size"]


def build_optimizer(model, prepared, cfg):
    if prepared.get("model_type") == "image":
        return torch.optim.AdamW(model.parameters(), lr=cfg.get("image_lr", cfg["lr"]), weight_decay=cfg.get("weight_decay", 0.0))
    return torch.optim.Adam(model.parameters(), lr=cfg["lr"])


def build_criterion(prepared, cfg):
    if prepared.get("model_type") == "image":
        return nn.CrossEntropyLoss(label_smoothing=cfg.get("label_smoothing", 0.0))
    return nn.CrossEntropyLoss()


def get_num_local_epochs(prepared, cfg):
    if prepared.get("model_type") == "image":
        return cfg.get("image_local_epochs", cfg["local_epochs"])
    return cfg["local_epochs"]


def build_scheduler(optimizer, prepared, cfg, epochs):
    if prepared.get("model_type") == "image" and epochs and epochs > 1:
        return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    return None


def apply_mixup(xb, yb, alpha=0.2):
    if alpha is None or alpha <= 0:
        return xb, yb, yb, 1.0
    lam = float(np.random.beta(alpha, alpha))
    index = torch.randperm(xb.size(0), device=xb.device)
    mixed_x = lam * xb + (1.0 - lam) * xb[index]
    return mixed_x, yb, yb[index], lam


def mixup_loss(criterion, logits, y_a, y_b, lam):
    return lam * criterion(logits, y_a) + (1.0 - lam) * criterion(logits, y_b)


def dirichlet_client_split(y, num_clients=5, alpha=0.6, min_size=32):
    y = np.asarray(y)
    classes = np.unique(y)
    while True:
        client_indices = [[] for _ in range(num_clients)]
        for cls in classes:
            cls_idx = np.where(y == cls)[0]
            np.random.shuffle(cls_idx)
            proportions = np.random.dirichlet(alpha=np.repeat(alpha, num_clients))
            cut_points = (np.cumsum(proportions)[:-1] * len(cls_idx)).astype(int)
            splits = np.split(cls_idx, cut_points)
            for client_id, split in enumerate(splits):
                client_indices[client_id].extend(split.tolist())
        sizes = [len(idx) for idx in client_indices]
        if min(sizes) >= min_size:
            return [np.array(sorted(idx)) for idx in client_indices]


def ensure_cached_client_indices(prepared, cfg):
    if prepared.get("model_type") == "image" and "client_subsets" in prepared:
        return None
    cache = prepared.setdefault("_client_split_cache", {})
    key = (cfg.get("num_clients", 5), cfg.get("dirichlet_alpha", 0.6), cfg.get("dirichlet_min_size", 32))
    if key not in cache:
        np_state = np.random.get_state()
        np.random.seed(SEED)
        cache[key] = dirichlet_client_split(prepared["y_train"], num_clients=cfg.get("num_clients", 5), alpha=cfg.get("dirichlet_alpha", 0.6), min_size=cfg.get("dirichlet_min_size", 32))
        np.random.set_state(np_state)
    return cache[key]


def build_fixed_client_loaders(prepared, cfg):
    batch_size = get_batch_size(prepared, cfg)
    if prepared.get("model_type") == "image" and "client_subsets" in prepared:
        client_loaders = [make_loader(subset, None, batch_size=batch_size, shuffle=True) for subset in prepared["client_subsets"]]
        client_sizes = [len(subset) for subset in prepared["client_subsets"]]
        return client_loaders, client_sizes

    client_indices = ensure_cached_client_indices(prepared, cfg)
    client_loaders = []
    client_sizes = []
    for idx in client_indices:
        client_loaders.append(make_loader(prepared["X_train"][idx], prepared["y_train"][idx], batch_size=batch_size, shuffle=True))
        client_sizes.append(len(idx))
    return client_loaders, client_sizes


def get_train_loader(prepared, cfg, shuffle=True):
    batch_size = get_batch_size(prepared, cfg)
    if prepared.get("model_type") == "image" and "train_dataset" in prepared:
        return make_loader(prepared["train_dataset"], None, batch_size=batch_size, shuffle=shuffle)
    return make_loader(prepared["X_train"], prepared["y_train"], batch_size=batch_size, shuffle=shuffle)


def get_ind_loader(prepared, cfg):
    batch_size = get_batch_size(prepared, cfg)
    if prepared.get("model_type") == "image" and "ind_test_dataset" in prepared:
        return make_loader(prepared["ind_test_dataset"], None, batch_size=batch_size, shuffle=False)
    return make_loader(prepared.get("X_ind_test", prepared["X_test"]), prepared.get("y_ind_test", prepared["y_test"]), batch_size=batch_size, shuffle=False)


def get_val_loader(prepared, cfg):
    batch_size = get_batch_size(prepared, cfg)
    if prepared.get("model_type") == "image" and "ood_val_dataset" in prepared:
        return make_loader(prepared["ood_val_dataset"], None, batch_size=batch_size, shuffle=False)
    if "X_val" in prepared and "y_val" in prepared:
        return make_loader(prepared["X_val"], prepared["y_val"], batch_size=batch_size, shuffle=False)
    return get_ind_loader(prepared, cfg)


def get_ood_loader(prepared, cfg):
    batch_size = get_batch_size(prepared, cfg)
    if prepared.get("model_type") == "image" and "ood_test_dataset" in prepared:
        return make_loader(prepared["ood_test_dataset"], None, batch_size=batch_size, shuffle=False)
    return make_loader(prepared.get("X_ood_test", prepared["X_test"]), prepared.get("y_ood_test", prepared["y_test"]), batch_size=batch_size, shuffle=False)

## Models and base federated training

A `FedAvg` model is trained first. That model is then used as the starting point for the unlearning methods.

In [5]:
class TabularNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward_features(self, x):
        return self.encoder(x)

    def forward(self, x):
        features = self.forward_features(x)
        logits = self.classifier(features)
        return logits, features


class ResNet18Classifier(nn.Module):
    def __init__(self, num_classes, input_channels=3, dropout_p=0.3):
        super().__init__()
        self.backbone = tv_models.resnet18(weights=tv_models.ResNet18_Weights.DEFAULT)
        if input_channels != 3:
            stem = self.backbone.conv1
            self.backbone.conv1 = nn.Conv2d(input_channels, stem.out_channels, kernel_size=stem.kernel_size, stride=stem.stride, padding=stem.padding, bias=False)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.dropout = nn.Dropout(dropout_p)
        self.classifier = nn.Linear(in_features, num_classes)

    def forward_features(self, x):
        if x.shape[-1] != 224 or x.shape[-2] != 224:
            x = F.interpolate(x, size=(224, 224), mode="bilinear", align_corners=False)
        x = self.backbone.conv1(x)
        x = self.backbone.bn1(x)
        x = self.backbone.relu(x)
        x = self.backbone.maxpool(x)
        x = self.backbone.layer1(x)
        x = self.backbone.layer2(x)
        x = self.backbone.layer3(x)
        x = self.backbone.layer4(x)
        x = self.backbone.avgpool(x)
        return torch.flatten(x, 1)

    def forward(self, x):
        features = self.forward_features(x)
        logits = self.classifier(self.dropout(features))
        return logits, features


def build_model(prepared, cfg):
    if prepared.get("model_type") == "image":
        return ResNet18Classifier(prepared["num_classes"], input_channels=prepared.get("input_channels", 3), dropout_p=cfg.get("image_dropout", 0.3))
    return TabularNet(prepared["input_dim"], cfg["hidden_dim"], prepared["num_classes"])


def model_num_bytes(model):
    return sum(p.numel() * p.element_size() for p in model.parameters())


def get_state(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def load_state(model, state):
    model.load_state_dict(state)


def clone_model_from_state(prepared, cfg, state):
    model = build_model(prepared, cfg).to(DEVICE)
    load_state(model, state)
    return model


def weighted_average_states(states, weights):
    out = copy.deepcopy(states[0])
    for key in out:
        out[key] = sum(w * states[i][key] for i, w in enumerate(weights))
    return out


def state_to_vector(state):
    return torch.cat([v.reshape(-1).float() for v in state.values()])


def evaluate_model(model, loader):
    model.eval()
    preds = []
    labels = []
    with torch.no_grad():
        for batch in loader:
            xb, yb, _ = unpack_batch(batch)
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            logits, _ = model(xb)
            preds.extend(logits.argmax(dim=1).cpu().numpy().tolist())
            labels.extend(yb.cpu().numpy().tolist())
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "precision_macro": precision_score(labels, preds, average="macro", zero_division=0),
        "recall_macro": recall_score(labels, preds, average="macro", zero_division=0),
    }


def tune_image_lr(prepared, cfg):
    if prepared.get("model_type") != "image":
        return cfg.get("image_lr", cfg["lr"])
    candidates = cfg.get("image_lr_candidates", [cfg.get("image_lr", cfg["lr"])])
    search_epochs = cfg.get("image_lr_search_epochs", 2)
    train_loader = get_train_loader(prepared, cfg, shuffle=True)
    val_loader = get_val_loader(prepared, cfg)
    best_lr = None
    best_acc = -1.0

    for lr in candidates:
        trial_cfg = dict(cfg)
        trial_cfg["image_lr"] = lr
        model = build_model(prepared, trial_cfg).to(DEVICE)
        optimizer = build_optimizer(model, prepared, trial_cfg)
        criterion = build_criterion(prepared, trial_cfg)
        scheduler = build_scheduler(optimizer, prepared, trial_cfg, search_epochs)

        for _ in range(search_epochs):
            model.train()
            for batch in train_loader:
                xb, yb, _ = unpack_batch(batch)
                xb = xb.to(DEVICE)
                yb = yb.to(DEVICE)
                optimizer.zero_grad()
                if trial_cfg.get("mixup_alpha", 0.0) > 0:
                    mixed_x, y_a, y_b, lam = apply_mixup(xb, yb, trial_cfg["mixup_alpha"])
                    logits, _ = model(mixed_x)
                    loss = mixup_loss(criterion, logits, y_a, y_b, lam)
                else:
                    logits, _ = model(xb)
                    loss = criterion(logits, yb)
                loss.backward()
                optimizer.step()
            if scheduler is not None:
                scheduler.step()

        val_acc = evaluate_model(model, val_loader)["accuracy"]
        print(f"lr={lr:.4g} -> val_acc={val_acc:.4f}")
        if val_acc > best_acc:
            best_acc = val_acc
            best_lr = lr
        del model
        release_memory()

    return best_lr


def local_train_epoch(model, loader, optimizer, criterion, prepared, cfg):
    model.train()
    for batch in loader:
        xb, yb, _ = unpack_batch(batch)
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)
        optimizer.zero_grad()
        if prepared.get("model_type") == "image" and cfg.get("mixup_alpha", 0.0) > 0:
            model_input, y_a, y_b, lam = apply_mixup(xb, yb, cfg["mixup_alpha"])
            logits, _ = model(model_input)
            loss = mixup_loss(criterion, logits, y_a, y_b, lam)
        else:
            logits, _ = model(xb)
            loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()


def run_fedavg_ext(prepared, cfg, record_history=False):
    client_loaders, client_sizes = build_fixed_client_loaders(prepared, cfg)
    global_model = build_model(prepared, cfg).to(DEVICE)
    criterion = build_criterion(prepared, cfg)
    history = []

    for _ in tqdm(range(cfg["rounds"]), desc="FedAvg", leave=False):
        base_state = get_state(global_model)
        client_states = []
        for loader in client_loaders:
            local_model = clone_model_from_state(prepared, cfg, base_state)
            optimizer = build_optimizer(local_model, prepared, cfg)
            local_epochs = get_num_local_epochs(prepared, cfg)
            scheduler = build_scheduler(optimizer, prepared, cfg, local_epochs)
            for _ in range(local_epochs):
                local_train_epoch(local_model, loader, optimizer, criterion, prepared, cfg)
                if scheduler is not None:
                    scheduler.step()
            client_states.append(get_state(local_model))
            del local_model

        weights = [size / sum(client_sizes) for size in client_sizes]
        next_state = weighted_average_states(client_states, weights)
        load_state(global_model, next_state)

        if record_history:
            history.append({
                "base_state": copy.deepcopy(base_state),
                "client_states": copy.deepcopy(client_states),
                "weights": list(weights),
            })

        release_memory()

    return {
        "method": "FedAvg",
        "model": global_model,
        "history": history,
    }

## Unlearning helpers

For `HR` and `BloodMNIST`, the forget and retain sets are built from array-based client splits. For `PACS`, the split is created from the chosen client subset.

In [6]:
class ForgetRetainDataset(torch.utils.data.Dataset):
    def __init__(self, forget=None, retain=None, anchor="forget"):
        self.forget = forget
        self.retain = retain
        self.anchor = anchor

    def __len__(self):
        if self.anchor == "forget":
            return len(self.forget)
        return len(self.retain)

    def __getitem__(self, idx):
        item = {}
        if self.anchor == "forget":
            item["forget"] = self.forget[idx]
            if self.retain is not None and len(self.retain) > 0:
                ridx = torch.randint(0, len(self.retain), (1,)).item()
                item["retain"] = self.retain[ridx]
        else:
            item["retain"] = self.retain[idx]
            if self.forget is not None and len(self.forget) > 0:
                fidx = torch.randint(0, len(self.forget), (1,)).item()
                item["forget"] = self.forget[fidx]
        return item


def build_client_unlearning_task(prepared, cfg, client_id=0, forget_fraction=0.2, seed=SEED):
    if "X_train" not in prepared or "y_train" not in prepared:
        raise ValueError("This helper expects an array-backed dataset.")

    client_indices = ensure_cached_client_indices(prepared, cfg)
    client_idx = np.array(client_indices[client_id])
    rng = np.random.default_rng(seed)
    shuffled = rng.permutation(client_idx)
    forget_count = max(1, int(len(shuffled) * forget_fraction))
    forget_idx = shuffled[:forget_count]
    retain_idx = shuffled[forget_count:]

    forget_ds = TensorDataset(torch.tensor(prepared["X_train"][forget_idx], dtype=torch.float32), torch.tensor(prepared["y_train"][forget_idx], dtype=torch.long))
    retain_ds = TensorDataset(torch.tensor(prepared["X_train"][retain_idx], dtype=torch.float32), torch.tensor(prepared["y_train"][retain_idx], dtype=torch.long))

    batch_size = get_batch_size(prepared, cfg)
    return {
        "forget_loader": DataLoader(forget_ds, batch_size=batch_size, shuffle=False),
        "retain_loader": DataLoader(retain_ds, batch_size=batch_size, shuffle=True),
    }


def build_pacs_client_unlearning_task(prepared, cfg, client_id=0, forget_fraction=0.2, seed=SEED):
    client_subset = prepared["client_subsets"][client_id]
    base_dataset = client_subset.dataset
    client_indices = np.array(client_subset.indices)

    rng = np.random.default_rng(seed)
    shuffled = client_indices.copy()
    rng.shuffle(shuffled)

    forget_count = max(1, int(len(shuffled) * forget_fraction))
    forget_indices = shuffled[:forget_count].tolist()
    retain_indices = shuffled[forget_count:].tolist()

    batch_size = get_batch_size(prepared, cfg)
    forget_subset = Subset(base_dataset, forget_indices)
    retain_subset = Subset(base_dataset, retain_indices)

    return {
        "forget_loader": DataLoader(forget_subset, batch_size=batch_size, shuffle=False),
        "retain_loader": DataLoader(retain_subset, batch_size=batch_size, shuffle=True),
    }


def evaluate_loss_and_accuracy(model, loader):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    total = 0
    correct = 0
    with torch.no_grad():
        for batch in loader:
            xb, yb = batch[:2]
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            logits, _ = model(xb)
            loss = criterion(logits, yb)
            total_loss += loss.item() * xb.size(0)
            total += xb.size(0)
            correct += (logits.argmax(dim=1) == yb).sum().item()
    return {
        "loss": total_loss / max(total, 1),
        "accuracy": correct / max(total, 1),
    }


def clone_base_model(prepared, cfg, base_model):
    model = build_model(prepared, cfg).to(DEVICE)
    load_state(model, get_state(base_model))
    return model


def split_unlearn(prepared, cfg, base_model, forget_loader, retain_loader, epochs=10):
    model = clone_base_model(prepared, cfg, base_model)
    optimizer = build_optimizer(model, prepared, cfg)

    for epoch in range(epochs):
        print(f"Split epoch {epoch + 1}/{epochs}")
        model.train()
        forget_iter = iter(forget_loader)
        retain_iter = iter(retain_loader)
        num_steps = max(len(forget_loader), len(retain_loader))

        for _ in range(num_steps):
            try:
                f_batch = next(forget_iter)
            except StopIteration:
                forget_iter = iter(forget_loader)
                f_batch = next(forget_iter)

            try:
                r_batch = next(retain_iter)
            except StopIteration:
                retain_iter = iter(retain_loader)
                r_batch = next(retain_iter)

            fx, fy, _ = unpack_batch(f_batch)
            rx, ry, _ = unpack_batch(r_batch)
            fx, fy = fx.to(DEVICE), fy.to(DEVICE)
            rx, ry = rx.to(DEVICE), ry.to(DEVICE)

            optimizer.zero_grad()
            forget_logits, _ = model(fx)
            forget_loss = F.cross_entropy(forget_logits, fy)
            (-forget_loss).backward()

            retain_logits, _ = model(rx)
            retain_loss = F.cross_entropy(retain_logits, ry)
            retain_loss.backward()
            optimizer.step()

    return model


def npo_unlearn(prepared, cfg, base_model, forget_loader, retain_loader, epochs=10, beta=0.5):
    model = clone_base_model(prepared, cfg, base_model)
    ref_model = clone_base_model(prepared, cfg, base_model)
    ref_model.eval()
    for p in ref_model.parameters():
        p.requires_grad = False

    optimizer = build_optimizer(model, prepared, cfg)
    retain_iter = iter(retain_loader)

    for epoch in range(epochs):
        print(f"NPO epoch {epoch + 1}/{epochs}")
        model.train()

        for forget_batch in forget_loader:
            fx, fy, _ = unpack_batch(forget_batch)
            fx, fy = fx.to(DEVICE), fy.to(DEVICE)

            try:
                retain_batch = next(retain_iter)
            except StopIteration:
                retain_iter = iter(retain_loader)
                retain_batch = next(retain_iter)

            rx, ry, _ = unpack_batch(retain_batch)
            rx, ry = rx.to(DEVICE), ry.to(DEVICE)

            optimizer.zero_grad()
            forget_logits, _ = model(fx)
            retain_logits, _ = model(rx)

            with torch.no_grad():
                ref_forget_logits, _ = ref_model(fx)

            cur_forget_ce = F.cross_entropy(forget_logits, fy, reduction="none")
            ref_forget_ce = F.cross_entropy(ref_forget_logits, fy, reduction="none")
            npo_loss = -torch.log(torch.sigmoid(beta * (cur_forget_ce - ref_forget_ce)) + 1e-8).mean()
            retain_loss = F.cross_entropy(retain_logits, ry)

            loss = npo_loss + retain_loss
            loss.backward()
            optimizer.step()

    del ref_model
    release_memory()
    return model


def gradient_ascent_unlearn(prepared, cfg, base_model, forget_loader, retain_loader, epochs=10, ascent_weight=2.0):
    model = clone_base_model(prepared, cfg, base_model)
    optimizer = build_optimizer(model, prepared, cfg)
    retain_iter = iter(retain_loader)

    for epoch in range(epochs):
        print(f"GA epoch {epoch + 1}/{epochs}")
        model.train()

        for forget_batch in forget_loader:
            fx, fy, _ = unpack_batch(forget_batch)
            fx, fy = fx.to(DEVICE), fy.to(DEVICE)

            try:
                retain_batch = next(retain_iter)
            except StopIteration:
                retain_iter = iter(retain_loader)
                retain_batch = next(retain_iter)

            rx, ry, _ = unpack_batch(retain_batch)
            rx, ry = rx.to(DEVICE), ry.to(DEVICE)

            optimizer.zero_grad()
            forget_logits, _ = model(fx)
            retain_logits, _ = model(rx)
            forget_loss = F.cross_entropy(forget_logits, fy)
            retain_loss = F.cross_entropy(retain_logits, ry)
            loss = -ascent_weight * forget_loss + retain_loss
            loss.backward()
            optimizer.step()

    return model


def amnesiac_unlearn(prepared, cfg, history, forgotten_client_id=0, rounds=5):
    if not history:
        return None
    model = build_model(prepared, cfg).to(DEVICE)
    usable_history = history[:min(rounds, len(history))]

    for round_info in usable_history:
        client_states = round_info["client_states"]
        weights = round_info["weights"]
        kept = [i for i in range(len(client_states)) if i != forgotten_client_id]
        kept_states = [client_states[i] for i in kept]
        kept_weights = torch.tensor([weights[i] for i in kept], dtype=torch.float32)
        kept_weights = (kept_weights / kept_weights.sum()).tolist()
        new_state = weighted_average_states(kept_states, kept_weights)
        load_state(model, new_state)

    return model


def zero_shot_unlearn(prepared, cfg, base_model, retain_loader, epochs=10):
    model = clone_base_model(prepared, cfg, base_model)
    optimizer = build_optimizer(model, prepared, cfg)

    for epoch in range(epochs):
        print(f"Zero-shot epoch {epoch + 1}/{epochs}")
        model.train()
        for batch in retain_loader:
            xb, yb, _ = unpack_batch(batch)
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            logits, _ = model(xb)
            loss = F.cross_entropy(logits, yb)
            loss.backward()
            optimizer.step()

    return model


def summarize_unlearning_model(dataset_name, prepared, cfg, task, model_name, model, rounds_used=None, epochs_used=None):
    return {
        "dataset": dataset_name,
        "model_variant": model_name,
        "rounds_used": rounds_used,
        "epochs_used": epochs_used,
        "forget_acc": evaluate_loss_and_accuracy(model, task["forget_loader"])["accuracy"],
        "retain_acc": evaluate_loss_and_accuracy(model, task["retain_loader"])["accuracy"],
        "ood_test_acc": evaluate_model(model, get_ood_loader(prepared, cfg))["accuracy"],
    }

## Build datasets

The next cell prepares the three datasets and tunes the image learning rate where needed.

In [7]:
prepared_datasets = {}
dataset_configs = {}

X_hr, y_hr = load_hr_dataset(HR_DATA_PATH)
prepared_datasets["hr_job_applicants"] = prepare_hr_dataset_country_shift(X_hr, y_hr)

pacs_prepared = load_pacs_dataset(PACS_ROOT, ood_domain=PACS_OOD_DOMAIN)
prepared_datasets[PACS_DATASET] = pacs_prepared

med_prepared = load_medmnist_dataset(MEDMNIST_DATASET)
prepared_datasets[MEDMNIST_DATASET] = med_prepared

for dataset_name, prepared in prepared_datasets.items():
    run_cfg = dict(CONFIG)
    if prepared.get("model_type") == "image":
        if dataset_name == PACS_DATASET:
            run_cfg["num_clients"] = len(prepared["client_groups"])
        else:
            run_cfg["num_clients"] = max(run_cfg.get("num_clients", 3), 5)
        run_cfg["image_lr"] = tune_image_lr(prepared, run_cfg)
    dataset_configs[dataset_name] = run_cfg

list(prepared_datasets.keys())

lr=0.0003 -> val_acc=0.6589
lr=0.001 -> val_acc=0.6293
lr=0.0003 -> val_acc=0.9620
lr=0.001 -> val_acc=0.9492


['hr_job_applicants', 'pacs', 'bloodmnist']

## Run unlearning for all datasets

This cell trains one `FedAvg` base model per dataset and evaluates the five unlearning methods.

In [8]:
UNLEARNING_EPOCHS = 10
FORGET_FRACTION = 0.20

all_unlearning_results = []

for dataset_name in ["hr_job_applicants", PACS_DATASET, MEDMNIST_DATASET]:
    prepared = prepared_datasets[dataset_name]
    cfg = dict(dataset_configs[dataset_name])

    print(f"\nRunning unlearning for {dataset_name} ...")
    base_fedavg_result = run_fedavg_ext(prepared, cfg, record_history=True)

    if dataset_name == PACS_DATASET:
        task = build_pacs_client_unlearning_task(prepared, cfg, client_id=0, forget_fraction=FORGET_FRACTION)
    else:
        task = build_client_unlearning_task(prepared, cfg, client_id=0, forget_fraction=FORGET_FRACTION)

    fedavg_rounds = len(base_fedavg_result["history"])

    baseline_row = summarize_unlearning_model(
        dataset_name, prepared, cfg, task,
        "FedAvg before unlearning",
        base_fedavg_result["model"],
        rounds_used=fedavg_rounds,
        epochs_used=None,
    )
    all_unlearning_results.append(baseline_row)

    split_model = split_unlearn(prepared, cfg, base_fedavg_result["model"], task["forget_loader"], task["retain_loader"], epochs=UNLEARNING_EPOCHS)
    all_unlearning_results.append(summarize_unlearning_model(dataset_name, prepared, cfg, task, "Split unlearning", split_model, rounds_used=None, epochs_used=UNLEARNING_EPOCHS))
    del split_model
    release_memory()

    npo_model = npo_unlearn(prepared, cfg, base_fedavg_result["model"], task["forget_loader"], task["retain_loader"], epochs=UNLEARNING_EPOCHS, beta=0.5)
    all_unlearning_results.append(summarize_unlearning_model(dataset_name, prepared, cfg, task, "NPO", npo_model, rounds_used=None, epochs_used=UNLEARNING_EPOCHS))
    del npo_model
    release_memory()

    ga_model = gradient_ascent_unlearn(prepared, cfg, base_fedavg_result["model"], task["forget_loader"], task["retain_loader"], epochs=UNLEARNING_EPOCHS, ascent_weight=2.0)
    all_unlearning_results.append(summarize_unlearning_model(dataset_name, prepared, cfg, task, "GA", ga_model, rounds_used=None, epochs_used=UNLEARNING_EPOCHS))
    del ga_model
    release_memory()

    amnesiac_model = amnesiac_unlearn(prepared, cfg, base_fedavg_result["history"], forgotten_client_id=0, rounds=min(5, fedavg_rounds))
    all_unlearning_results.append(summarize_unlearning_model(dataset_name, prepared, cfg, task, "Amnesiac", amnesiac_model, rounds_used=min(5, fedavg_rounds), epochs_used=None))
    del amnesiac_model
    release_memory()

    zero_shot_model = zero_shot_unlearn(prepared, cfg, base_fedavg_result["model"], task["retain_loader"], epochs=UNLEARNING_EPOCHS)
    all_unlearning_results.append(summarize_unlearning_model(dataset_name, prepared, cfg, task, "Zero-shot", zero_shot_model, rounds_used=None, epochs_used=UNLEARNING_EPOCHS))
    del zero_shot_model
    del base_fedavg_result, task
    release_memory()

all_unlearning_df = pd.DataFrame(all_unlearning_results)
all_unlearning_df


Running unlearning for hr_job_applicants ...


FedAvg:   0%|          | 0/10 [00:00<?, ?it/s]

Split epoch 1/10
Split epoch 2/10
Split epoch 3/10
Split epoch 4/10
Split epoch 5/10
Split epoch 6/10
Split epoch 7/10
Split epoch 8/10
Split epoch 9/10
Split epoch 10/10
NPO epoch 1/10
NPO epoch 2/10
NPO epoch 3/10
NPO epoch 4/10
NPO epoch 5/10
NPO epoch 6/10
NPO epoch 7/10
NPO epoch 8/10
NPO epoch 9/10
NPO epoch 10/10
GA epoch 1/10
GA epoch 2/10
GA epoch 3/10
GA epoch 4/10
GA epoch 5/10
GA epoch 6/10
GA epoch 7/10
GA epoch 8/10
GA epoch 9/10
GA epoch 10/10
Zero-shot epoch 1/10
Zero-shot epoch 2/10
Zero-shot epoch 3/10
Zero-shot epoch 4/10
Zero-shot epoch 5/10
Zero-shot epoch 6/10
Zero-shot epoch 7/10
Zero-shot epoch 8/10
Zero-shot epoch 9/10
Zero-shot epoch 10/10

Running unlearning for pacs ...


FedAvg:   0%|          | 0/10 [00:00<?, ?it/s]

Split epoch 1/10
Split epoch 2/10
Split epoch 3/10
Split epoch 4/10
Split epoch 5/10
Split epoch 6/10
Split epoch 7/10
Split epoch 8/10
Split epoch 9/10
Split epoch 10/10
NPO epoch 1/10
NPO epoch 2/10
NPO epoch 3/10
NPO epoch 4/10
NPO epoch 5/10
NPO epoch 6/10
NPO epoch 7/10
NPO epoch 8/10
NPO epoch 9/10
NPO epoch 10/10
GA epoch 1/10
GA epoch 2/10
GA epoch 3/10
GA epoch 4/10
GA epoch 5/10
GA epoch 6/10
GA epoch 7/10
GA epoch 8/10
GA epoch 9/10
GA epoch 10/10
Zero-shot epoch 1/10
Zero-shot epoch 2/10
Zero-shot epoch 3/10
Zero-shot epoch 4/10
Zero-shot epoch 5/10
Zero-shot epoch 6/10
Zero-shot epoch 7/10
Zero-shot epoch 8/10
Zero-shot epoch 9/10
Zero-shot epoch 10/10

Running unlearning for bloodmnist ...


FedAvg:   0%|          | 0/10 [00:00<?, ?it/s]

Split epoch 1/10
Split epoch 2/10
Split epoch 3/10
Split epoch 4/10
Split epoch 5/10
Split epoch 6/10
Split epoch 7/10
Split epoch 8/10
Split epoch 9/10
Split epoch 10/10
NPO epoch 1/10
NPO epoch 2/10
NPO epoch 3/10
NPO epoch 4/10
NPO epoch 5/10
NPO epoch 6/10
NPO epoch 7/10
NPO epoch 8/10
NPO epoch 9/10
NPO epoch 10/10
GA epoch 1/10
GA epoch 2/10
GA epoch 3/10
GA epoch 4/10
GA epoch 5/10
GA epoch 6/10
GA epoch 7/10
GA epoch 8/10
GA epoch 9/10
GA epoch 10/10
Zero-shot epoch 1/10
Zero-shot epoch 2/10
Zero-shot epoch 3/10
Zero-shot epoch 4/10
Zero-shot epoch 5/10
Zero-shot epoch 6/10
Zero-shot epoch 7/10
Zero-shot epoch 8/10
Zero-shot epoch 9/10
Zero-shot epoch 10/10


,dataset,model_variant,rounds_used,epochs_used,forget_acc,retain_acc,ood_test_acc
0,hr_job_applicants,FedAvg before unlearning,10.0,NaN,1.000000,0.999708,0.649485
1,hr_job_applicants,Split unlearning,NaN,10.0,0.001753,0.026001,0.517581
2,hr_job_applicants,NPO,NaN,10.0,0.998831,1.000000,0.536036
3,hr_job_applicants,GA,NaN,10.0,0.000000,0.000219,0.517128
4,hr_job_applicants,Amnesiac,5.0,NaN,0.604325,0.591002,0.665739
5,hr_job_applicants,Zero-shot,NaN,10.0,1.000000,1.000000,0.574824
6,pacs,FedAvg before unlearning,10.0,NaN,1.000000,1.000000,0.690076
7,pacs,Split unlearning,NaN,10.0,0.000000,0.390542,0.047837
8,pacs,NPO,NaN,10.0,0.703364,0.982456,0.270738
9,pacs,GA,NaN,10.0,0.006116,0.292906,0.154707


## View the results

In [9]:
display(all_unlearning_df.style.format({
    "forget_acc": "{:.4f}",
    "retain_acc": "{:.4f}",
    "ood_test_acc": "{:.4f}",
}, na_rep="N/A"))

,dataset,model_variant,rounds_used,epochs_used,forget_acc,retain_acc,ood_test_acc
0,hr_job_applicants,FedAvg before unlearning,10.000000,N/A,1.0000,0.9997,0.6495
1,hr_job_applicants,Split unlearning,N/A,10.000000,0.0018,0.0260,0.5176
2,hr_job_applicants,NPO,N/A,10.000000,0.9988,1.0000,0.5360
3,hr_job_applicants,GA,N/A,10.000000,0.0000,0.0002,0.5171
4,hr_job_applicants,Amnesiac,5.000000,N/A,0.6043,0.5910,0.6657
5,hr_job_applicants,Zero-shot,N/A,10.000000,1.0000,1.0000,0.5748
6,pacs,FedAvg before unlearning,10.000000,N/A,1.0000,1.0000,0.6901
7,pacs,Split unlearning,N/A,10.000000,0.0000,0.3905,0.0478
8,pacs,NPO,N/A,10.000000,0.7034,0.9825,0.2707
9,pacs,GA,N/A,10.000000,0.0061,0.2929,0.1547


## Separate tables for each dataset

In [10]:
hr_unlearning_df = all_unlearning_df[all_unlearning_df["dataset"] == "hr_job_applicants"].reset_index(drop=True)
pacs_unlearning_df = all_unlearning_df[all_unlearning_df["dataset"] == PACS_DATASET].reset_index(drop=True)
medmnist_unlearning_df = all_unlearning_df[all_unlearning_df["dataset"] == MEDMNIST_DATASET].reset_index(drop=True)

print("HR")
display(hr_unlearning_df)
print("PACS")
display(pacs_unlearning_df)
print("BloodMNIST")
display(medmnist_unlearning_df)

HR


,dataset,model_variant,rounds_used,epochs_used,forget_acc,retain_acc,ood_test_acc
0,hr_job_applicants,FedAvg before unlearning,10.0,NaN,1.000000,0.999708,0.649485
1,hr_job_applicants,Split unlearning,NaN,10.0,0.001753,0.026001,0.517581
2,hr_job_applicants,NPO,NaN,10.0,0.998831,1.000000,0.536036
3,hr_job_applicants,GA,NaN,10.0,0.000000,0.000219,0.517128
4,hr_job_applicants,Amnesiac,5.0,NaN,0.604325,0.591002,0.665739
5,hr_job_applicants,Zero-shot,NaN,10.0,1.000000,1.000000,0.574824


PACS


,dataset,model_variant,rounds_used,epochs_used,forget_acc,retain_acc,ood_test_acc
0,pacs,FedAvg before unlearning,10.0,NaN,1.000000,1.000000,0.690076
1,pacs,Split unlearning,NaN,10.0,0.000000,0.390542,0.047837
2,pacs,NPO,NaN,10.0,0.703364,0.982456,0.270738
3,pacs,GA,NaN,10.0,0.006116,0.292906,0.154707
4,pacs,Amnesiac,5.0,NaN,0.932722,0.935164,0.696183
5,pacs,Zero-shot,NaN,10.0,0.951070,0.992372,0.623410


BloodMNIST


,dataset,model_variant,rounds_used,epochs_used,forget_acc,retain_acc,ood_test_acc
0,bloodmnist,FedAvg before unlearning,10.0,NaN,0.992,0.986083,0.688395
1,bloodmnist,Split unlearning,NaN,10.0,0.000,0.500994,0.266589
2,bloodmnist,NPO,NaN,10.0,0.744,1.000000,0.341713
3,bloodmnist,GA,NaN,10.0,0.000,0.274354,0.263081
4,bloodmnist,Amnesiac,5.0,NaN,0.968,0.974155,0.789243
5,bloodmnist,Zero-shot,NaN,10.0,0.968,1.000000,0.585794


## Export to Excel

In [11]:
output_path = Path("core/outputs/unlearning_all_datasets.xlsx")
output_path.parent.mkdir(parents=True, exist_ok=True)

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    all_unlearning_df.to_excel(writer, sheet_name="All", index=False)
    hr_unlearning_df.to_excel(writer, sheet_name="HR", index=False)
    pacs_unlearning_df.to_excel(writer, sheet_name="PACS", index=False)
    medmnist_unlearning_df.to_excel(writer, sheet_name="BloodMNIST", index=False)

print(f"Saved to: {output_path.resolve()}")

Saved to: /Users/shashank.ajmani/Desktop/NTU Assigments/Data Privacy/Learning/DomainFL/core/core/outputs/unlearning_all_datasets.xlsx
